## Setup

Just importing everything needed for parsing, modeling and evaluation.

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.model_selection import train_test_split, GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

## Load and inspect

Reading in the isolate data and checking its shape and missing values first, before deciding what to filter out (vaccine strains, monophasic isolates, etc.).

In [ ]:
df = pd.read_csv("VIR_Salmonella_isolates.tsv", sep="\t")
print(df.shape)
print(df.columns.tolist())
print(df[["Serovar", "Isolation type", "AMR genotypes", "Stress genotypes", "Virulence genotypes"]].isna().sum())

In [ ]:
print(df["Serovar"].value_counts(dropna=False))

## Cleaning up Typhimurium labels

Dropping isolates labelled as vaccine strains, and keeping only the ones whose computed serotype is Typhimurium — this also gets rid of the monophasic variant.

In [ ]:
vaccine_mask = df["Serovar"].str.contains("vaccine", case=False, na=False)
print("vaccine strain rows:", vaccine_mask.sum())
df = df.loc[~vaccine_mask].reset_index(drop=True)

typhi_declared_mask = df["Serovar"].str.contains("typhimurium", case=False, na=False)
print("declared Typhimurium rows:", typhi_declared_mask.sum())
df = df.loc[typhi_declared_mask].reset_index(drop=True)

def extract_computed_serotype(s):
    m = re.search(r"serotype=(.*)$", str(s))
    return m.group(1) if m else None

df["computed_serotype"] = df["Computed types"].apply(extract_computed_serotype)
print(df["computed_serotype"].value_counts(dropna=False))

In [ ]:
pure_mask = df["computed_serotype"] == "Typhimurium"
print("pure Typhimurium n:", pure_mask.sum())
df = df.loc[pure_mask].reset_index(drop=True)
print(df["Isolation type"].value_counts())

In [ ]:
df.head()

In [ ]:
print(df["AMR genotypes"].dropna().iloc[:5].tolist())
print()
print(df["Stress genotypes"].dropna().iloc[:5].tolist())
print()
print(df["Virulence genotypes"].dropna().iloc[:5].tolist())

## Parsing gene calls

Turning each isolate's messy gene-call text into a clean set of genes it actually carries.

In [ ]:
AMR_QUALIFIERS = {"COMPLETE", "POINT", "PARTIAL_END_OF_CONTIG"}
STRESS_VIR_QUALIFIERS = {"COMPLETE", "PARTIAL_END_OF_CONTIG"}

def parse_genes(entry, included_qualifiers):
    genes = set()
    for token in entry.split(","):
        if "=" not in token:
            continue
        gene, qualifier = token.rsplit("=", 1)
        if qualifier in included_qualifiers:
            genes.add(gene)
    return genes

df["amr_genes"] = df["AMR genotypes"].apply(lambda e: {"AMR__" + g for g in parse_genes(e, AMR_QUALIFIERS)})
df["stress_genes"] = df["Stress genotypes"].apply(lambda e: {"STRESS__" + g for g in parse_genes(e, STRESS_VIR_QUALIFIERS)})
df["vir_genes"] = df["Virulence genotypes"].apply(lambda e: {"VIR__" + g for g in parse_genes(e, STRESS_VIR_QUALIFIERS)})
df["all_genes"] = [a | b | c for a, b, c in zip(df["amr_genes"], df["stress_genes"], df["vir_genes"])]

gene_counts = df["all_genes"].apply(len)
print(gene_counts.describe())
print("isolates with zero genes:", (gene_counts == 0).sum())

In [ ]:
print(df["Isolation type"].unique())
print(df["Isolation source"].value_counts().head(15))



In [ ]:
print(pd.crosstab(df["Isolation source"] == "human", df["Isolation type"]))

In [ ]:
print(df.loc[(df["Isolation source"] != "human") & (df["Isolation type"] == "clinical"), "Isolation source"].value_counts().head(20))

In [ ]:
print(df.loc[df["Isolation source"] == "Slaughter and breeding farm", ["Host", "Isolation type", "Location"]])
print()
print(df.loc[df["Isolation source"] == "Uncertain", ["Host", "Isolation type", "Location"]])

In [ ]:
print(df["Host"].unique())
print()
print(pd.crosstab(df["Host"] == "Homo sapiens", df["Isolation type"]))

In [ ]:
print(df.loc[(df["Isolation type"] == "clinical") & (df["Host"] != "Homo sapiens"), ["Host", "Isolation source", "Location"]])
print()
print(df["Host"].isna().sum())

## Verifying clinical vs environmental labels

Cross-checking isolation source and host info to make sure the clinical/environmental labels actually hold up, and dropping the isolates that don't.

In [ ]:
non_human_clinical_mask = (df["Isolation type"] == "clinical") & (
    (df["Host"] != "Homo sapiens") | df["Host"].isna()
)
print("dropping:", non_human_clinical_mask.sum())
df = df.loc[~non_human_clinical_mask].reset_index(drop=True)
print(df["Isolation type"].value_counts())

In [ ]:
print(df.loc[df["Isolation type"] == "environmental/other", "Host"].value_counts(dropna=False))
print()
print(df.loc[df["Isolation type"] == "environmental/other", "Isolation source"].value_counts(dropna=False).head(20))

## Building the feature matrix

One-hot encoding the gene sets into a proper feature matrix, and turning the label into 1 (clinical) / 0 (environmental).

In [ ]:
mlb = MultiLabelBinarizer()
X = pd.DataFrame(mlb.fit_transform(df["all_genes"]), columns=mlb.classes_, index=df.index)
y = (df["Isolation type"] == "clinical").astype(int)

print(X.shape)
print(y.value_counts())

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
gene_freq = X.sum(axis=0).sort_values()
print(gene_freq.head(15))
print()
print((gene_freq <= 2).sum(), "genes present in <=2 isolates out of", X.shape[1], "total")

## Baseline Random Forest

First try — plain train/test split and a Random Forest, just to see where we stand.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(Xtr, ytr)

yp = rf.predict(Xte)
ypr = rf.predict_proba(Xte)[:, 1]

acc = accuracy_score(yte, yp)
auroc = roc_auc_score(yte, ypr)
tn, fp, fn, tp = confusion_matrix(yte, yp).ravel()
print(f"Accuracy: {acc:.3f}  AUROC: {auroc:.3f}  Sensitivity: {tp/(tp+fn):.3f}  Specificity: {tn/(tn+fp):.3f}")

## Checking for leakage

Checking whether near-identical genomes (same SNP cluster) are leaking between train and test, and how skewed the clusters are — i.e. whether one clone is dominating.

In [ ]:
train_clusters = set(df.loc[Xtr.index, "SNP cluster"].dropna())
test_clusters = df.loc[Xte.index, "SNP cluster"]

leaked = test_clusters.isin(train_clusters)
print(f"{leaked.sum()} / {len(test_clusters)} test isolates ({leaked.mean()*100:.1f}%) share a SNP cluster with a training isolate")

In [ ]:
sizes = df["SNP cluster"].value_counts()
print(f"{len(sizes)} clusters total, largest 5: {sizes.head(5).tolist()}")
print(f"top cluster = {sizes.iloc[0]} isolates ({sizes.iloc[0]/len(df)*100:.1f}% of the dataset)")

## Fixing the leakage — GroupKFold

Redoing the evaluation with GroupKFold so an entire SNP cluster stays on one side of the split. This is the actual trustworthy score.

In [ ]:
def group_kfold_eval(n_splits=5, n_estimators=300):
    groups = df["SNP cluster"].copy()
    missing = groups.isna()
    groups.loc[missing] = ["SINGLETON_" + str(i) for i in df.index[missing]]

    gkf = GroupKFold(n_splits=n_splits)
    aurocs, accs = [], []
    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        Xtr_k, Xte_k = X.iloc[train_idx], X.iloc[test_idx]
        ytr_k, yte_k = y.iloc[train_idx], y.iloc[test_idx]
        rf_k = RandomForestClassifier(n_estimators=n_estimators, class_weight="balanced", random_state=42, n_jobs=-1)
        rf_k.fit(Xtr_k, ytr_k)
        aurocs.append(roc_auc_score(yte_k, rf_k.predict_proba(Xte_k)[:, 1]))
        accs.append(accuracy_score(yte_k, rf_k.predict(Xte_k)))

    print("AUROC per fold =", [round(a, 3) for a in aurocs])
    print(f"Mean AUROC: {np.mean(aurocs):.3f} +/- {np.std(aurocs):.3f}")
    print(f"Mean Accuracy: {np.mean(accs):.3f} +/- {np.std(accs):.3f}")

group_kfold_eval()

## Gene count baseline

Testing whether the model is just picking up on "more genes = more likely clinical" rather than actually learning from which genes are present.

In [ ]:
df["total_gene_count"] = df["all_genes"].apply(len)
X_count_only = df[["total_gene_count"]]

def group_kfold_eval_features(X_custom, y_custom, label, n_splits=5, n_estimators=300):
    groups = df["SNP cluster"].copy()
    missing = groups.isna()
    groups.loc[missing] = ["SINGLETON_" + str(i) for i in df.index[missing]]

    gkf = GroupKFold(n_splits=n_splits)
    aurocs = []
    for train_idx, test_idx in gkf.split(X_custom, y_custom, groups=groups):
        Xtr_k, Xte_k = X_custom.iloc[train_idx], X_custom.iloc[test_idx]
        ytr_k, yte_k = y_custom.iloc[train_idx], y_custom.iloc[test_idx]
        rf_k = RandomForestClassifier(n_estimators=n_estimators, class_weight="balanced", random_state=42, n_jobs=-1)
        rf_k.fit(Xtr_k, ytr_k)
        aurocs.append(roc_auc_score(yte_k, rf_k.predict_proba(Xte_k)[:, 1]))

    print(f"{label}: AUROC per fold = {[round(a,3) for a in aurocs]}")
    print(f"  Mean AUROC: {np.mean(aurocs):.3f} +/- {np.std(aurocs):.3f}")

group_kfold_eval_features(X_count_only, y, "gene COUNT only (no identity)")

## Feature importance and direction

Checking which genes the model relies on most, and whether each one leans clinical or environmental.

In [ ]:
rf_full = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42, n_jobs=-1)
rf_full.fit(X, y)

importances = pd.Series(rf_full.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.head(20))

In [ ]:
top20 = importances.head(20).index
rates = pd.DataFrame({
    "clinical_rate": X.loc[y == 1, top20].mean(),
    "environmental_rate": X.loc[y == 0, top20].mean(),
})
rates["importance"] = importances[top20]
rates["diff"] = rates["clinical_rate"] - rates["environmental_rate"]
print(rates.sort_values("importance", ascending=False))

## Clonal concentration check

Checking whether a gene is basically tied to one or two clusters, which could skew the clinical vs environmental prediction. Most genes turned out to be spread across a lot of different clusters — only two of them were concentrated in a handful.

In [ ]:
ybtp_pos = df.loc[X["VIR__ybtP"] == 1]
print("ybtP positive isolates:", len(ybtp_pos))
print(ybtp_pos["SNP cluster"].value_counts())
print()

for gene in top20:
    carriers = df.loc[X[gene] == 1, "SNP cluster"]
    n_isolates = len(carriers)
    n_clusters = carriers.nunique()
    top_share = carriers.value_counts(normalize=True).iloc[0] if n_isolates else float("nan")
    print(f"{gene:15s} n={n_isolates:4d}  distinct_clusters={n_clusters:4d}  top_cluster_share={top_share:.2f}")

## Null check

Shuffling the labels and rerunning everything, just to make sure the model isn't faking a good score on nonsense.

In [ ]:
np.random.seed(0)
y_shuffled = pd.Series(np.random.permutation(y.values), index=y.index)

def group_kfold_eval_custom(y_custom, label, n_splits=5, n_estimators=300):
    groups = df["SNP cluster"].copy()
    missing = groups.isna()
    groups.loc[missing] = ["SINGLETON_" + str(i) for i in df.index[missing]]

    gkf = GroupKFold(n_splits=n_splits)
    aurocs = []
    for train_idx, test_idx in gkf.split(X, y_custom, groups=groups):
        Xtr_k, Xte_k = X.iloc[train_idx], X.iloc[test_idx]
        ytr_k, yte_k = y_custom.iloc[train_idx], y_custom.iloc[test_idx]
        if yte_k.nunique() < 2:
            continue
        rf_k = RandomForestClassifier(n_estimators=n_estimators, class_weight="balanced", random_state=42, n_jobs=-1)
        rf_k.fit(Xtr_k, ytr_k)
        aurocs.append(roc_auc_score(yte_k, rf_k.predict_proba(Xte_k)[:, 1]))

    print(f"{label}: AUROC per fold = {[round(a,3) for a in aurocs]}")
    print(f"  Mean AUROC: {np.mean(aurocs):.3f} +/- {np.std(aurocs):.3f}")

group_kfold_eval_custom(y_shuffled, "Typhimurium (pure) - SHUFFLED labels")

## Figures

Feature importance/direction, the overall validation summary, the ROC curve, and the clonal concentration check, plotted out.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

top_n = 15
top_genes = importances.head(top_n).index

plot_df = pd.DataFrame({
    "importance": importances[top_genes],
    "diff": X.loc[y == 1, top_genes].mean() - X.loc[y == 0, top_genes].mean(),
}).sort_values("importance")  # ascending so most important ends up at top of the barh

colors = ["#d62728" if d > 0 else "#1f77b4" for d in plot_df["diff"]]

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(plot_df.index, plot_df["diff"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Presence rate difference (clinical - environmental)")
ax.set_title("Top predictive genes: direction and magnitude")
ax.legend(handles=[
    Patch(facecolor="#d62728", label="More common in clinical"),
    Patch(facecolor="#1f77b4", label="More common in environmental/other"),
], loc="lower right")

plt.tight_layout()
plt.savefig("fig1_feature_importance_direction.png", dpi=200)
plt.show()

In [ ]:
def group_kfold_auroc(X_custom, y_custom, n_splits=5, n_estimators=300):
    groups = df["SNP cluster"].copy()
    missing = groups.isna()
    groups.loc[missing] = ["SINGLETON_" + str(i) for i in df.index[missing]]

    gkf = GroupKFold(n_splits=n_splits)
    aurocs = []
    for train_idx, test_idx in gkf.split(X_custom, y_custom, groups=groups):
        Xtr_k, Xte_k = X_custom.iloc[train_idx], X_custom.iloc[test_idx]
        ytr_k, yte_k = y_custom.iloc[train_idx], y_custom.iloc[test_idx]
        if yte_k.nunique() < 2:
            continue
        rf_k = RandomForestClassifier(n_estimators=n_estimators, class_weight="balanced", random_state=42, n_jobs=-1)
        rf_k.fit(Xtr_k, ytr_k)
        aurocs.append(roc_auc_score(yte_k, rf_k.predict_proba(Xte_k)[:, 1]))
    return aurocs

honest_aurocs = group_kfold_auroc(X, y)
count_aurocs = group_kfold_auroc(X_count_only, y)
null_aurocs = group_kfold_auroc(X, y_shuffled)

labels = ["Naive random split\n(leaky)", "Honest\n(GroupKFold)", "Gene count only\n(no identity)", "Shuffled labels\n(null)"]
means = [auroc, np.mean(honest_aurocs), np.mean(count_aurocs), np.mean(null_aurocs)]
errors = [0, np.std(honest_aurocs), np.std(count_aurocs), np.std(null_aurocs)]
colors = ["#7f7f7f", "#2ca02c", "#ff7f0e", "#d62728"]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, means, yerr=errors, capsize=5, color=colors)
ax.axhline(0.5, color="black", linestyle="--", linewidth=1, label="Chance level")
ax.set_ylim(0, 1.0)
ax.set_ylabel("AUROC")
ax.set_title("Validation story: from inflated to honest")

for bar, mean, err in zip(bars, means, errors):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + err + 0.02, f"{mean:.3f}", ha="center")

ax.legend()
plt.tight_layout()
plt.savefig("fig2_validation_story.png", dpi=200)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

def group_kfold_oof_predictions(X_custom, y_custom, n_splits=5, n_estimators=300):
    groups = df["SNP cluster"].copy()
    missing = groups.isna()
    groups.loc[missing] = ["SINGLETON_" + str(i) for i in df.index[missing]]

    gkf = GroupKFold(n_splits=n_splits)
    oof_true, oof_proba = [], []
    for train_idx, test_idx in gkf.split(X_custom, y_custom, groups=groups):
        Xtr_k, Xte_k = X_custom.iloc[train_idx], X_custom.iloc[test_idx]
        ytr_k, yte_k = y_custom.iloc[train_idx], y_custom.iloc[test_idx]
        rf_k = RandomForestClassifier(n_estimators=n_estimators, class_weight="balanced", random_state=42, n_jobs=-1)
        rf_k.fit(Xtr_k, ytr_k)
        oof_true.extend(yte_k.tolist())
        oof_proba.extend(rf_k.predict_proba(Xte_k)[:, 1].tolist())
    return np.array(oof_true), np.array(oof_proba)

y_true_oof, y_proba_oof = group_kfold_oof_predictions(X, y)

fpr, tpr, _ = roc_curve(y_true_oof, y_proba_oof)
auroc_oof = roc_auc_score(y_true_oof, y_proba_oof)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, color="#2ca02c", linewidth=2, label=f"Honest GroupKFold (AUROC = {auroc_oof:.3f})")
ax.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1, label="Chance")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curve — out-of-fold predictions (Typhimurium, pure)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("fig3_roc_curve.png", dpi=200)
plt.show()

In [ ]:
top_genes = importances.head(20).index

stats = []
for gene in top_genes:
    carriers = df.loc[X[gene] == 1, "SNP cluster"]
    n_isolates = len(carriers)
    top_share = carriers.value_counts(normalize=True).iloc[0] if n_isolates else np.nan
    stats.append({"gene": gene.split("__")[1], "top_cluster_share": top_share})

stats_df = pd.DataFrame(stats).set_index("gene").sort_values("top_cluster_share")

colors = ["#d62728" if s > 0.3 else "#7f7f7f" for s in stats_df["top_cluster_share"]]

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(stats_df.index, stats_df["top_cluster_share"], color=colors)
ax.axvline(0.3, color="black", linestyle="--", linewidth=1, label="Concentration threshold")
ax.set_xlabel("Fraction of gene's carriers found in a single SNP cluster")
ax.set_title("Is this gene a broad signal or one lineage's fingerprint?")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("fig4_clonal_concentration.png", dpi=200)
plt.show()

## Comparing other models

Running Decision Tree, Logistic Regression, Gradient Boosting, KNN and Bernoulli Naive Bayes through the same setup, to see how they stack up against Random Forest.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

def group_kfold_auroc_model(model_fn, X_custom, y_custom, n_splits=5):
    groups = df["SNP cluster"].copy()
    missing = groups.isna()
    groups.loc[missing] = ["SINGLETON_" + str(i) for i in df.index[missing]]

    gkf = GroupKFold(n_splits=n_splits)
    aurocs = []
    for train_idx, test_idx in gkf.split(X_custom, y_custom, groups=groups):
        Xtr_k, Xte_k = X_custom.iloc[train_idx], X_custom.iloc[test_idx]
        ytr_k, yte_k = y_custom.iloc[train_idx], y_custom.iloc[test_idx]
        model = model_fn()
        model.fit(Xtr_k, ytr_k)
        aurocs.append(roc_auc_score(yte_k, model.predict_proba(Xte_k)[:, 1]))
    return aurocs

dt_aurocs = group_kfold_auroc_model(lambda: DecisionTreeClassifier(class_weight="balanced", random_state=42), X, y)

print("Decision Tree: AUROC per fold =", [round(a, 3) for a in dt_aurocs])
print(f"Mean AUROC: {np.mean(dt_aurocs):.3f} +/- {np.std(dt_aurocs):.3f}")
print(f"\nFor comparison, Random Forest: Mean AUROC: {np.mean(honest_aurocs):.3f} +/- {np.std(honest_aurocs):.3f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

models = {
    "Logistic Regression": lambda: LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "Gradient Boosting": lambda: GradientBoostingClassifier(random_state=42),
    "K-Nearest Neighbors": lambda: KNeighborsClassifier(n_neighbors=15),
}

results = {"Decision Tree": dt_aurocs, "Random Forest": honest_aurocs}
for name, model_fn in models.items():
    aurocs = group_kfold_auroc_model(model_fn, X, y)
    results[name] = aurocs
    print(f"{name}: Mean AUROC = {np.mean(aurocs):.3f} +/- {np.std(aurocs):.3f}")

In [ ]:
def group_kfold_auroc_weighted(model_fn, X_custom, y_custom, n_splits=5):
    groups = df["SNP cluster"].copy()
    missing = groups.isna()
    groups.loc[missing] = ["SINGLETON_" + str(i) for i in df.index[missing]]

    gkf = GroupKFold(n_splits=n_splits)
    aurocs = []
    for train_idx, test_idx in gkf.split(X_custom, y_custom, groups=groups):
        Xtr_k, Xte_k = X_custom.iloc[train_idx], X_custom.iloc[test_idx]
        ytr_k, yte_k = y_custom.iloc[train_idx], y_custom.iloc[test_idx]

        class_counts = ytr_k.value_counts()
        weight_map = {cls: len(ytr_k) / (2 * count) for cls, count in class_counts.items()}
        sw = ytr_k.map(weight_map)

        model = model_fn()
        model.fit(Xtr_k, ytr_k, sample_weight=sw)
        aurocs.append(roc_auc_score(yte_k, model.predict_proba(Xte_k)[:, 1]))
    return aurocs

gb_weighted_aurocs = group_kfold_auroc_weighted(lambda: GradientBoostingClassifier(random_state=42), X, y)
print("Gradient Boosting (class-weighted): Mean AUROC =", f"{np.mean(gb_weighted_aurocs):.3f} +/- {np.std(gb_weighted_aurocs):.3f}")
print("Gradient Boosting (unweighted, from before): Mean AUROC =", f"{np.mean(results['Gradient Boosting']):.3f} +/- {np.std(results['Gradient Boosting']):.3f}")

In [ ]:
lr_unweighted_aurocs = group_kfold_auroc_model(lambda: LogisticRegression(max_iter=1000, random_state=42), X, y)

print("Logistic Regression (unweighted): Mean AUROC =", f"{np.mean(lr_unweighted_aurocs):.3f} +/- {np.std(lr_unweighted_aurocs):.3f}")
print("Logistic Regression (class_weight='balanced', from before): Mean AUROC =", f"{np.mean(results['Logistic Regression']):.3f} +/- {np.std(results['Logistic Regression']):.3f}")

In [ ]:
from sklearn.naive_bayes import BernoulliNB

nb_aurocs = group_kfold_auroc_model(lambda: BernoulliNB(), X, y)
results["Bernoulli Naive Bayes"] = nb_aurocs

print(f"Bernoulli Naive Bayes: Mean AUROC = {np.mean(nb_aurocs):.3f} +/- {np.std(nb_aurocs):.3f}")